# Agentic Curriculum Builder (Base Prototype)


In [ ]:
# Setup AI API using OpenAI

from dotenv import load_dotenv
from openai import OpenAI
import os

# Step Zero: Create .env file with API key
# To access OpenAI LLMs you need to sign up and get an API key
# Then you need to create a file in this working directory called `.env`
# The file content should be on a single line:
# OPENAI_API_KEY = "YOUR_KEY"
# (Do not include the # in your line)

# Get API key from .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
# If you need packages, here is the code to install some packages
# !pip install langgraph langchain langchain-openai
# !pip install langchain_anthropic langchain_google_genai

import operator
from typing import Annotated, TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.checkpoint.memory import MemorySaver


In [ ]:
# 0. Initialize Memory for the agent
memory = MemorySaver()

# 1a. Define the Shared State for the agent
# These are the "notes" that get passed between nodes.
class AgentState(TypedDict):
    initial_topic: str
    brainstorm: str
    draft: str
    # This 'operator.add' is the secret sauce for parallel work
    reviewer_feedback: Annotated[List[str], operator.add]
    editor_feedback: str
    revised_draft: str
    final_content: str

    revision_count: int

# 1b. Define the AI models available for the agent
#     temperature changes how 'creative' a model will be.
llm_nano      = ChatOpenAI(model="gpt-5-nano", temperature=0.2)
llm_nano_high = ChatOpenAI(model="gpt-5-nano", temperature=0.8)
llm_5         = ChatOpenAI(model="gpt-5",      temperature=0.2)
llm_5_high    = ChatOpenAI(model="gpt-5",      temperature=0.8)


In [ ]:
# 2. Set up the agent components

initial_topic = "Masked Auto Encoder (MAE) for GeoAI"

# Central dictionary to store node configurations
NODE_CONFIGS = {
    "brainstormer": {
        "system_role": "Creative Brainstorming Educator",
        "system_instruction": "Generate diverse, innovative, and highly relevant angles, topics, and core ideas based on the user's prompt. Provide a structured list of 5 distinct concepts that can serve as the foundation for a complete draft.",
        "human_role": "",
        "human_instruction": "Please brainstorm ideas and content angles for the following topic:\n\nTopic: ",
        "llm": llm_nano_high,
        "output_key": "brainstorm"
    },
    "writer": {
        "system_role": "Expert Geospatial Data Science Instructional Designer",
        "system_instruction": 
            "Draft a highly focused, 9-minute hands-on Jupyter Notebook lab exercise for " +
            "the Convergence Curriculum for Geospatial Data Science. Structure your response " +
            "to clearly delineate 'Markdown Cells' (for learning objectives, context, and instructions) " +
            "and 'Code Cells' (for Python code using relevant libraries like GeoPandas, Shapely, or Folium). " +
            "Pacing is critical: keep the scope strictly limited to what a student can read, execute, " +
            "and understand within exactly 9 minutes.",
        "human_role": "Curriculum Developer",
        "human_instruction": "Draft the Jupyter Notebook lab exercise based on the topic brainstormed ideas.\n\n",
        "llm": llm_nano,
        "output_key": "draft"
    },
    "reviewer_a": {
        "system_role": "Senior Geospatial Data Scientist and Scholar. " +
                       "You are a skeptical academic from GIS and Geography. Review this for geographic theory gaps and misunderstanding of GIS concepts and technologies",
        "system_instruction":  "Keep comments succinct.",
        "human_role": "Curriculum Reviewer",
        "human_instruction": "Review this draft.",
        "llm": llm_nano,
        "output_key": "reviewer_feedback"
    },
    "reviewer_b": {
        "system_role": "Senior Computer Scientist. " +
                       "You are a skeptical academic from Computer Science. Review this for technical accuracy and feasibility.",
        "system_instruction":  "Keep comments succinct.",
        "human_role": "Curriculum Reviewer",
        "human_instruction": "Review this draft.",
        "llm": llm_nano,
        "output_key": "reviewer_feedback"
    },
    "reviewer_c": {
        "system_role": "Senior Education Scholar." +
                       "You are a skeptical academic from Education with a specialization in Ed Tech." +
                       "Review this for pedagogical content and ensure it will be relevant and approachable for beginner learners",
        "system_instruction":  "Keep comments succinct.",
        "human_role": "Curriculum Reviewer",
        "human_instruction": "Review this draft.",
        "llm": llm_nano,
        "output_key": "reviewer_feedback"
    },
    "editor": {
        "system_role": "Senior Editor",
        "system_instruction": "Make suggestions to fix grammar and improve writing based on feedback." +
                              "Highlight the main themes and give the writer clear directions on what to fix. " +
                              "Disentangle any conflicting advice. Keep comments and directions descriptive and clear.",
        "human_role": "Editor",
        "human_instruction": "Draft writing and feedback to incorporate.",
        "llm": llm_5,
        "output_key": "editor_feedback"
    },
    "reviser": {
        "system_role": "Author and Expert Geospatial Data Science Instructional Designer",
        "system_instruction": "Rewrite the draft strictly following the editor's instructions. " +
                              "Be critical in your changes. Maintain the original intent but fix all issues.",
        "human_role": "Author",
        "human_instruction": "Editors Instructions with reviews and original draft.",
        "llm": llm_5,
        "output_key": "revised_draft"
    },    
    "copywriter": {
        "system_role": "Copywriter and Expert Writer",
        "system_instruction": "You are a master copyeditor. Do NOT change the facts or the scientific arguments. " +
                              "Tighten the prose, fix grammar, improve flow, and make it punchy.\n\n",
        "human_role": "Copywriter",
        "human_instruction": "Revised draft:",
        "llm": llm_5,
        "output_key": "final_content"
    }    
}


In [ ]:
# 3. Set up the system to make the agent work

def make_messages(config: dict, state: dict, extra: str) -> list:
    """
    Takes a node config, the graph state, and additional text, returning formatted messages for the LLM.
    """
    # 1. Build the System Message
    system_content = f"Role: {config['system_role']}\nInstructions: {config['system_instruction']}"
    system_message = SystemMessage(content=system_content)

    # 2. Build the Human Message
    human_content = f"{config['human_role']}\nInstructions: {config['human_instruction']}\n\n {extra}"
    human_message = HumanMessage(content=human_content)
    
    # Tried and failed at thi
    #return [system_content, human_content]
    
    # 3. Return the exact list format that llm.invoke() wants
    return [system_message, human_message]
    

In [ ]:
# 4. Make the nodes

def brainstormer_node(state: dict):
    config = NODE_CONFIGS["brainstormer"]

    extra = state['initial_topic']
    
    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    #messages[1].append({state['initial_topic']})
    #messages[1] += state['initial_topic']
    #msg = [SystemMessage(content=messages[0]),HumanMessage(content=messages[1])]
    #response = config['llm'].invoke(msg)
    
    # Call the LLM
    response = config['llm'].invoke(messages)

    # Update the state
    return {config['output_key']: [response.content]}

def writer_node(state: dict):
    config = NODE_CONFIGS["writer"]

    extra = f"Topic: {state['initial_topic']} \nBrainstorm ideas: {state['brainstorm']}"
    
    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    
    # Call the LLM
    response = config['llm'].invoke(messages)
    
    # Update the state
    return {config['output_key']: [response.content]}

def reviewer_a(state: dict):
    config = NODE_CONFIGS["reviewer_a"]

    extra = state['draft']

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    
    # Call the LLM
    response = config['llm'].invoke(messages)
    
    # Update the state
    return {config['output_key']: [response.content]}

def reviewer_b(state: dict):
    config = NODE_CONFIGS["reviewer_b"]

    extra = state['draft']
    
    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    #messages[1].append({state['draft']})
    
    # Call the LLM
    response = config['llm'].invoke(messages)
    
    # Update the state
    return {config['output_key']: [response.content]}

def reviewer_c(state: dict):
    config = NODE_CONFIGS["reviewer_c"]
    
    extra = state['draft']
    
    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    #messages[1].append({state['draft']})
    
    # Call the LLM
    response = config['llm'].invoke(messages)
    
    # Update the state
    return {config['output_key']: [response.content]}


def editor_node(state: dict):
    config = NODE_CONFIGS["editor"]

    all_reviews = "\n\n".join(state['reviewer_feedback'])

    extra = f"Draft: {state['draft']}\n\n Reviews: {all_reviews}"
        
    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    
    # Call the LLM
    response = config['llm'].invoke(messages)
    
    # Update the state
    return {config['output_key']: [response.content]}

def reviser_node(state: dict):
    config = NODE_CONFIGS["reviser"]

    all_reviews = "\n\n".join(state['reviewer_feedback'])

    extra =  f"Draft: {state['draft']}\n\n"
    extra += f"Reviews: {all_reviews}\n\n"
    extra += f"Editor Feedback: {state['editor_feedback']}\n\n"
    
    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    
    # Call the LLM
    response = config['llm'].invoke(messages)
    
    # Update the state
    return {config['output_key']: [response.content]}

def copywriter_node(state: dict):
    config = NODE_CONFIGS["copywriter"]
    
    extra = f"Revised Draft: {state['revised_draft']}\n\n"

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    
    # Call the LLM
    response = config['llm'].invoke(messages)
    
    # Update the state
    return {config['output_key']: [response.content]}


In [ ]:
# Create a workflow using AgentState
def make_workflow():
    workflow = StateGraph(AgentState)
    return workflow
    
# Add nodes to our agent workflow
def make_nodes(workflow):
    workflow.add_node("brainstormer", brainstormer_node) # Brainstorming
    workflow.add_node("writer",       writer_node)       # Drafts
    
    # 3 reviewers
    workflow.add_node("reviewer_a", reviewer_a)
    workflow.add_node("reviewer_b", reviewer_b)
    workflow.add_node("reviewer_c", reviewer_c)
    
    workflow.add_node("editor",     editor_node)
    workflow.add_node("reviser",    reviser_node)        
    workflow.add_node("copywriter", copywriter_node)

    return workflow

# Define the edges and the flow of the agent workflow
def make_connections(workflow):

    # The start
    workflow.set_entry_point("brainstormer")

    workflow.add_edge("brainstormer", "writer")
    
    # Multiple reviewers
    # FAN-OUT: Writer goes to all reviewers at once
    workflow.add_edge("writer", "reviewer_a")
    workflow.add_edge("writer", "reviewer_b")
    workflow.add_edge("writer", "reviewer_c")
    
    # FAN-IN: All reviewers point to the Editor
    # LangGraph automatically waits for ALL THREE to finish before starting the Editor.
    workflow.add_edge("reviewer_a", "editor")
    workflow.add_edge("reviewer_b", "editor")
    workflow.add_edge("reviewer_c", "editor")

    # Bring them all back and revise
    workflow.add_edge("editor", "reviser")
    workflow.add_edge("reviser", "copywriter")

    # The end
    workflow.add_edge("copywriter", END)
    
    return workflow

def make_app():
    workflow = make_workflow()
    workflow = make_nodes(workflow)
    workflow = make_connections(workflow)
    
    # Compile the workflow
    app = workflow.compile(checkpointer=memory)

    return app

In [ ]:
# Kick off the process
inputs = {"initial_topic": "", "draft": "", 
          "reviewer_feedback": [], "editor_feedback": "", 
          "revised_draft": "", "final_content": "",
          "revision_count": 0}
config = {"configurable": {"thread_id": "1"}}

# Set the initial topic
inputs['initial_topic'] = initial_topic

# Make the agent app
app = make_app()

for output in app.stream(inputs, config):
    for key, value in output.items():
        print(f"Finished node: {key}")

# Final Result
final_state = app.get_state(config)
print("\n--- FINAL POLISHED CONTENT ---")
#print("FS",final_state)
#print("FSV",final_state.values)

final_values = app.get_state(config).values
if "final_content" in final_values:
        content = final_values["final_content"]
#print(final_state.values["final_polish"])

print(final_state.values.keys())
#print(final_state.values['draft'])
print(final_state.values['final_content'])

In [ ]:
# Save the final contents so they are not lost.
with open("output.txt", "w") as file:
    file.write(str(final_state.values['final_content']))

In [ ]:
import nbformat

# Convert the contents to a lab exercise in Jupyter Notebooks.
def simple_text_to_jupyter(raw_text, output_filename="geospatial_lab.ipynb"):

    raw_text = raw_text.replace('\\n', '\n')
    raw_text = raw_text.replace('\\\'', '\'')
    
    # 1. Create a blank notebook using the official library
    nb = nbformat.v4.new_notebook()

    # 2. THE TRICK: Replace the markers with a unique splitting word
    text = raw_text.replace("Markdown Cell", "SPLIT_HERE_MARKDOWN")
    text = text.replace("Code Cell", "SPLIT_HERE_CODE")
    
    # 3. Break the text into a list of chunks based on our unique word
    chunks = text.split("SPLIT_HERE_")
    
    for chunk in chunks:
        # 4. If it's a markdown chunk, clean the text and add a markdown cell
        if chunk.startswith("MARKDOWN"):
            content = chunk.replace("MARKDOWN", "").strip()
            if content:
                nb.cells.append(nbformat.v4.new_markdown_cell(content))
                
        # 5. If it's a code chunk, clean the text and add a code cell
        elif chunk.startswith("CODE"):
            content = chunk.replace("CODE", "#").strip()
            
            # (Optional but helpful) strip out backticks if the LLM added them
            content = content.replace("```python", "").replace("```", "").strip()
            
            if content:
                nb.cells.append(nbformat.v4.new_code_cell(content))

    # 6. Save the file using the library's built-in writer
    with open(output_filename, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)
        
    print(f"Saved to {output_filename} successfully!")

final_text = str(final_state.values['final_content'])

simple_text_to_jupyter(final_text)